# End-to-End Motion Retargeting Pipeline

This notebook runs the full pipeline from start to finish:

- video -> MediaPipe -> BVH
- image -> procedural human mesh OBJ + rig metadata
- mesh -> Blender rigging
- BVH -> Blender retargeting
- animated scene -> MP4 render

The notebook uses the procedural mesh backend because it is the mesh generator currently wired into the rigging and retargeting pipeline.


## Environment

Install the Python dependencies in the notebook kernel environment if needed:

```bash
pip install mediapipe opencv-python numpy
```

Make sure `blender` is available in `PATH` before running the Blender stages.
System `ffmpeg` is recommended for the PNG-sequence fallback path, but native Blender movie output may still work without it.

For segmented PNGs or illustrations such as `zelda.png`, the mesh stage may use the built-in silhouette fallback instead of MediaPipe Pose. That is expected and no longer causes the notebook to fail.


In [1]:
from pathlib import Path
import importlib
import json
from pprint import pprint
import sys

from IPython.display import Video, display

PROJECT_ROOT = Path("/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common import DEFAULT_BODY_HEIGHT_M, ensure_directory
from main import extract_first_frame
from motion_capture import capture_motion_to_bvh
from rigging import run_rigging_stage
from retargeting import run_retargeting_stage
from blender.render_runner import run_render_stage
from pipeline_validation import (
    validate_blend_file,
    validate_bvh,
    validate_mesh_metadata_json,
    validate_mp4,
    validate_obj,
)

import mesh_generation.procedural_human_mesh as procedural_human_mesh
importlib.reload(procedural_human_mesh)
generate_humanoid_mesh_from_image = procedural_human_mesh.generate_humanoid_mesh_from_image

def show_validation(label, result):
    print(f"{label}: ok={result.ok}")
    if result.warnings:
        print("Warnings:")
        for warning in result.warnings:
            print(" -", warning)
    if result.errors:
        print("Errors:")
        for error in result.errors:
            print(" -", error)
        raise RuntimeError(f"{label} validation failed")
    if result.details:
        pprint(result.details)

PROJECT_ROOT


PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3')

In [2]:
VIDEO_PATH = PROJECT_ROOT / "taekwondo-1.mp4"
IMAGE_PATH = PROJECT_ROOT / "zelda.png"
OUTPUT_DIR = ensure_directory(PROJECT_ROOT / "outputs" / "notebook_demo")
BLENDER_EXE = "blender"

TARGET_HEIGHT_M = DEFAULT_BODY_HEIGHT_M
FRAME_STEP = 1
RESOLUTION_X = 1920
RESOLUTION_Y = 1080

if not VIDEO_PATH.exists():
    raise FileNotFoundError(VIDEO_PATH)

if not IMAGE_PATH.exists():
    IMAGE_PATH = extract_first_frame(VIDEO_PATH, OUTPUT_DIR / "mesh_input.png")

print("Video:", VIDEO_PATH)
print("Image:", IMAGE_PATH)
print("Output directory:", OUTPUT_DIR)


Video: /home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/taekwondo-1.mp4
Image: /home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/zelda.png
Output directory: /home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo


In [3]:
bvh_path = OUTPUT_DIR / "motion.bvh"
overlay_path = OUTPUT_DIR / "motion_overlay.mp4"
mesh_path = OUTPUT_DIR / "generated_human.obj"
metadata_path = OUTPUT_DIR / "generated_human.json"
rigged_blend_path = OUTPUT_DIR / "rigged_scene.blend"
rig_report_path = OUTPUT_DIR / "rig_report.json"
animated_blend_path = OUTPUT_DIR / "animated_scene.blend"
retarget_report_path = OUTPUT_DIR / "retarget_report.json"
render_path = OUTPUT_DIR / "retargeted_animation.mp4"
manifest_path = OUTPUT_DIR / "pipeline_manifest.json"

{
    "bvh_path": bvh_path,
    "overlay_path": overlay_path,
    "mesh_path": mesh_path,
    "metadata_path": metadata_path,
    "rigged_blend_path": rigged_blend_path,
    "rig_report_path": rig_report_path,
    "animated_blend_path": animated_blend_path,
    "retarget_report_path": retarget_report_path,
    "render_path": render_path,
    "manifest_path": manifest_path,
}


{'bvh_path': PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/motion.bvh'),
 'overlay_path': PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/motion_overlay.mp4'),
 'mesh_path': PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/generated_human.obj'),
 'metadata_path': PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/generated_human.json'),
 'rigged_blend_path': PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/rigged_scene.blend'),
 'rig_report_path': PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/rig_report.json'),
 'animated_blend_path': PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/animated_scene.blend'),
 'retarget_report_path': PosixPath('/home/root_james

## 1. Extract Motion to BVH


In [4]:
motion_result = capture_motion_to_bvh(
    video_path=VIDEO_PATH,
    bvh_path=bvh_path,
    overlay_path=overlay_path,
    scale=1.0,
    frame_step=FRAME_STEP,
)

print(f"BVH written to: {motion_result.bvh_path}")
print(f"Overlay video: {motion_result.overlay_path}")
print(f"Frame count: {motion_result.frame_count}")
print(f"FPS: {motion_result.fps:.3f}")
if motion_result.warnings:
    print("Warnings:")
    for warning in motion_result.warnings:
        print(" -", warning)
if motion_result.foot_contact_frames:
    print("Foot contacts:", motion_result.foot_contact_frames)

show_validation("motion.bvh", validate_bvh(bvh_path))


I0000 00:00:1773203732.588751  520865 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1773203732.603424  521025 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 550.163.01), renderer: NVIDIA GeForce RTX 4070 Ti SUPER/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


BVH written to: /home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/motion.bvh
Overlay video: /home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/motion_overlay.mp4
Frame count: 72
FPS: 25.000
Warnings:
 - interpolated missing samples for LeftWrist
 - interpolated missing samples for LeftHand
 - interpolated missing samples for RightElbow
 - interpolated missing samples for RightWrist
 - interpolated missing samples for RightHand
Foot contacts: {'LeftFoot': 0, 'RightFoot': 0}
motion.bvh: ok=True
{'frame_count': 72, 'frame_time': 0.04, 'joint_count': 21, 'root_name': 'Hips'}


## 2. Generate the Human Mesh


In [5]:
mesh_result = generate_humanoid_mesh_from_image(
    image_path=IMAGE_PATH,
    output_obj_path=mesh_path,
    metadata_path=metadata_path,
    target_height_m=TARGET_HEIGHT_M,
)

mesh_metadata = json.loads(metadata_path.read_text())

print(f"Mesh OBJ: {mesh_result.mesh_path}")
print(f"Rig metadata: {mesh_result.metadata_path}")
print(f"Estimated height: {mesh_result.estimated_height_m:.3f} m")
print(f"Measurement backend: {mesh_metadata.get('measurement_backend', 'unknown')}")

show_validation("generated_human.obj", validate_obj(mesh_path))
show_validation("generated_human.json", validate_mesh_metadata_json(metadata_path))


Mesh OBJ: /home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/generated_human.obj
Rig metadata: /home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/generated_human.json
Estimated height: 1.700 m
Measurement backend: silhouette_fallback
generated_human.obj: ok=True
{'face_count': 8520, 'vertex_count': 4572}
generated_human.json: ok=True
{'estimated_height_m': 1.7, 'measurement_backend': 'silhouette_fallback'}


/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/mesh_generation/procedural_human_mesh.py:443: RuntimeWarning: zelda.png looks like a segmented illustration or cutout; using silhouette fallback measurements instead.
  warnings.warn(


## 3. Rig the Mesh in Blender


In [6]:
run_rigging_stage(
    blender_executable=BLENDER_EXE,
    mesh_obj_path=mesh_path,
    metadata_json_path=metadata_path,
    blend_output_path=rigged_blend_path,
    report_output_path=rig_report_path,
    project_root=PROJECT_ROOT,
)

show_validation("rigged_scene.blend", validate_blend_file(rigged_blend_path))
if rig_report_path.exists():
    pprint(json.loads(rig_report_path.read_text()))

rigged_blend_path


rigged_scene.blend: ok=True
{'size_bytes': 246505}
{'armature_name': 'TargetRig',
 'blend_out': '/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/rigged_scene.blend',
 'bone_count': 21,
 'bone_naming_convention': 'schema_joint_name',
 'estimated_height_m': 1.7,
 'mesh_obj': '/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/generated_human.obj',
 'mesh_object_name': 'GeneratedHuman',
 'metadata_json': '/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/generated_human.json',
 'scale_factor': 5.034994983654537,
 'status': 'ok',
 'vertex_count': 4572,
 'warnings': []}


PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/rigged_scene.blend')

## 4. Retarget the BVH Motion


In [7]:
run_retargeting_stage(
    blender_executable=BLENDER_EXE,
    input_blend_path=rigged_blend_path,
    metadata_json_path=metadata_path,
    bvh_path=bvh_path,
    blend_output_path=animated_blend_path,
    report_output_path=retarget_report_path,
    project_root=PROJECT_ROOT,
)

show_validation("animated_scene.blend", validate_blend_file(animated_blend_path))
if retarget_report_path.exists():
    pprint(json.loads(retarget_report_path.read_text()))

animated_blend_path


animated_scene.blend: ok=True
{'size_bytes': 503196}
{'axis_multipliers': [1.0, 1.0, 1.0],
 'blend_in': '/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/rigged_scene.blend',
 'blend_out': '/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/animated_scene.blend',
 'bvh': '/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/motion.bvh',
 'frame_count': 72,
 'frame_time': 0.04,
 'joint_mapping': {'Chest': 'Chest',
                   'Head': 'Head',
                   'Hips': 'Hips',
                   'LeftAnkle': 'LeftAnkle',
                   'LeftElbow': 'LeftElbow',
                   'LeftFoot': 'LeftFoot',
                   'LeftHand': 'LeftHand',
                   'LeftHip': 'LeftHip',
                   'LeftKnee': 'LeftKnee',
                   'LeftShoulder': 'LeftShoulder',
                   'LeftWrist': 'LeftWrist',
                   'Neck': 'Neck',
                   '

PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/animated_scene.blend')

## 5. Render the Final MP4


In [8]:
run_render_stage(
    blender_executable=BLENDER_EXE,
    input_blend_path=animated_blend_path,
    metadata_json_path=metadata_path,
    output_video_path=render_path,
    project_root=PROJECT_ROOT,
    resolution_x=RESOLUTION_X,
    resolution_y=RESOLUTION_Y,
)

show_validation("retargeted_animation.mp4", validate_mp4(render_path))

render_path


retargeted_animation.mp4: ok=True
{'codec_name': 'h264', 'height': '1080', 'size_bytes': 453688, 'width': '1920'}


PosixPath('/home/root_james/Projects/pilot_AXON/mesh_motion_extraction_v3/outputs/notebook_demo/retargeted_animation.mp4')

## 6. Preview the Outputs


In [9]:
generated_files = sorted(path.name for path in OUTPUT_DIR.iterdir())
generated_files


['animated_scene.blend',
 'animated_scene.blend1',
 'generated_human.json',
 'generated_human.obj',
 'motion.bvh',
 'motion_overlay.mp4',
 'retarget_report.json',
 'retargeted_animation.mp4',
 'retargeted_animation_frames',
 'rig_report.json',
 'rigged_scene.blend',
 'rigged_scene.blend1']

In [10]:
if manifest_path.exists():
    pprint(json.loads(manifest_path.read_text()))
else:
    print("pipeline_manifest.json is only written by main.py. This notebook runs stages directly.")


pipeline_manifest.json is only written by main.py. This notebook runs stages directly.


In [11]:
if overlay_path.exists():
    display(Video(str(overlay_path), embed=True, html_attributes="controls"))

if render_path.exists():
    display(Video(str(render_path), embed=True, html_attributes="controls"))
